# Clase 29: Introducción a redes neuronales

En esta clase vamos a entender qué es una red neuronal, cómo aprende y cómo usar `MLPClassifier` de scikit-learn.

> **Idea central:** Una red neuronal aprende representaciones progresivamente más complejas de los datos, pasando la información por capas de transformación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('seaborn-v0_8')
np.random.seed(42)
print('Librerías cargadas')

## 1. Cómo funciona una neurona — cálculo paso a paso

Una neurona artificial hace una operación muy simple:

```
z = x1·w1 + x2·w2 + ... + xn·wn + b
salida = f(z)
```

Donde `f` es la función de activación (sigmoid, ReLU, etc.)

In [ ]:
# Calculamos la salida de una neurona manualmente
# Entrada: [nota_matematicas=7.5, asistencia=0.85, edad=23]
x = np.array([7.5, 0.85, 23.0])

# Pesos (inicialmente aleatorios, se aprenden durante el entrenamiento)
w = np.array([0.4, 0.3, -0.1])

# Bias
b = 0.5

# Suma ponderada
z = np.dot(x, w) + b
print(f'z = {x[0]}×{w[0]} + {x[1]}×{w[1]} + {x[2]}×{w[2]} + {b}')
print(f'z = {z:.3f}')

# Función de activación Sigmoid
sigmoid = 1 / (1 + np.exp(-z))
print(f'\nSigmoid(z) = {sigmoid:.4f}')
print(f'Interpretación: {sigmoid:.1%} de probabilidad de aprobar')

# Función de activación ReLU
relu = max(0, z)
print(f'\nReLU(z) = {relu:.4f}')

## 2. Visualizar funciones de activación

In [ ]:
z_vals = np.linspace(-6, 6, 300)

sigmoid_vals = 1 / (1 + np.exp(-z_vals))
relu_vals = np.maximum(0, z_vals)
tanh_vals = np.tanh(z_vals)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(z_vals, sigmoid_vals, color='#4C72B0', linewidth=2.5)
axes[0].set_title('Sigmoid: salida entre 0 y 1', fontsize=11)
axes[0].set_xlabel('z')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='0.5')
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.4)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(z_vals, relu_vals, color='#55A868', linewidth=2.5)
axes[1].set_title('ReLU: max(0, z)', fontsize=11)
axes[1].set_xlabel('z')
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.4)
axes[1].grid(True, alpha=0.3)

axes[2].plot(z_vals, tanh_vals, color='#C44E52', linewidth=2.5)
axes[2].set_title('Tanh: salida entre -1 y 1', fontsize=11)
axes[2].set_xlabel('z')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.4)
axes[2].axvline(0, color='gray', linestyle='--', alpha=0.4)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Funciones de activación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('ReLU es la más usada en capas ocultas porque:')
print('- Es simple de calcular')
print('- No tiene el problema del gradiente que desaparece')
print('- Funciona muy bien en práctica')

## 3. Preparar datos

**Punto crítico:** Las redes neuronales necesitan que los datos estén normalizados. Si una variable va de 0 a 10 y otra de 18 a 35, la segunda va a dominar los cálculos de los pesos.

In [ ]:
# Crear dataset de estudiantes
n = 400
cursos = np.random.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
nota_mate = np.where(
    cursos == 'A', np.random.normal(7.5, 1.2, n),
    np.where(cursos == 'B', np.random.normal(7.0, 1.3, n), np.random.normal(5.8, 1.5, n))
).clip(1, 10)
nota_lengua = np.where(
    cursos == 'A', np.random.normal(7.2, 1.1, n),
    np.where(cursos == 'B', np.random.normal(6.8, 1.2, n), np.random.normal(5.5, 1.4, n))
).clip(1, 10)
asistencia = np.where(cursos == 'C', np.random.uniform(0.55, 0.85, n), np.random.uniform(0.70, 0.99, n))
edades = np.random.randint(18, 35, size=n)
prob = (nota_mate * 0.4 + nota_lengua * 0.4 + asistencia * 10 * 0.2) / 10
aprobado = (np.random.uniform(0, 1, n) < prob).astype(int)

df = pd.DataFrame({
    'nota_matematicas': nota_mate.round(1),
    'nota_lengua': nota_lengua.round(1),
    'asistencia': asistencia.round(2),
    'edad': edades,
    'aprobado': aprobado
})

features = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
X = df[features]
y = df['aprobado']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalización: media=0, desviación estándar=1
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print('Datos preparados:')
print(f'  Train: {X_train_sc.shape}')
print(f'  Test:  {X_test_sc.shape}')
print()
print('Antes de normalizar (primeras 2 filas):')
print(X_train.head(2).to_string())
print()
print('Después de normalizar (primeras 2 filas):')
print(pd.DataFrame(X_train_sc[:2], columns=features).round(3).to_string())

## 4. Entrenar el primer MLPClassifier

In [ ]:
# Red con 2 capas ocultas: 100 y 50 neuronas
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),  # arquitectura de la red
    activation='relu',             # función de activación en capas ocultas
    solver='adam',                 # algoritmo de optimización
    max_iter=300,                  # número máximo de epochs
    learning_rate_init=0.001,      # tasa de aprendizaje inicial
    random_state=42
)

mlp.fit(X_train_sc, y_train)

predicciones = mlp.predict(X_test_sc)
acc = accuracy_score(y_test, predicciones)

print(f'Precisión del MLP (100, 50): {acc:.2%}')
print()
print('Arquitectura de la red:')
print(f'  Capa de entrada: {X_train_sc.shape[1]} neuronas (una por variable)')
print(f'  Capa oculta 1:  100 neuronas (ReLU)')
print(f'  Capa oculta 2:  50 neuronas (ReLU)')
print(f'  Capa de salida: 1 neurona (Sigmoid — probabilidad)')
print()
print(classification_report(y_test, predicciones, target_names=['Reprobado', 'Aprobado']))

## 5. Curva de aprendizaje — cómo fue aprendiendo el modelo

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(mlp.loss_curve_, color='#4C72B0', linewidth=2)
plt.title('Curva de aprendizaje del MLP\nCómo disminuye el error a lo largo del entrenamiento')
plt.xlabel('Epoch (iteración sobre el dataset completo)')
plt.ylabel('Pérdida (error)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Error en la epoch 1:    {mlp.loss_curve_[0]:.4f}')
print(f'Error en la epoch {len(mlp.loss_curve_)//2}:  {mlp.loss_curve_[len(mlp.loss_curve_)//2]:.4f}')
print(f'Error en la epoch final: {mlp.loss_curve_[-1]:.4f}')
print()
print('La curva debe bajar y estabilizarse. Si no baja, hay un problema de')
print('aprendizaje. Si baja y luego sube, hay overfitting.')

## 6. Comparar MLP vs Decision Tree vs Random Forest

In [ ]:
import time

modelos = {
    'MLP (100, 50) — necesita escalar': (mlp, X_test_sc),
    'Decision Tree (max_depth=5)': (DecisionTreeClassifier(max_depth=5, random_state=42), X_test),
    'Random Forest (100 árboles)': (RandomForestClassifier(n_estimators=100, random_state=42), X_test),
}

resultados = {}

# Ya tenemos el MLP entrenado, entrenamos los demás
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

for nombre, modelo in [('Decision Tree', tree), ('Random Forest', rf)]:
    inicio = time.time()
    modelo.fit(X_train, y_train)
    tiempo = time.time() - inicio
    acc = accuracy_score(y_test, modelo.predict(X_test))
    resultados[nombre] = {'precision': acc, 'tiempo': tiempo}

acc_mlp = accuracy_score(y_test, mlp.predict(X_test_sc))
resultados['MLP (100, 50)'] = {'precision': acc_mlp, 'tiempo': None}

print('=== Comparación de modelos ===')
print(f'{"Modelo":<25} {"Precisión":>10}')
print('-' * 37)
for nombre, info in sorted(resultados.items(), key=lambda x: x[1]['precision'], reverse=True):
    tiempo_str = f"{info['tiempo']:.3f}s" if info['tiempo'] else 'N/A'
    print(f'{nombre:<25} {info["precision"]:>10.2%}')

print()
print('OBSERVACIÓN IMPORTANTE:')
print('En datos tabulares pequeños-medianos, Random Forest suele ganar o empatar')
print('con MLP, con mucho menos esfuerzo de configuración.')
print()
print('Las redes neuronales brillan en: imágenes, texto, audio, series de tiempo')
print('y cuando tienes millones de datos.')

## 7. El efecto de la normalización

Veamos qué pasa si olvidamos normalizar antes de entrenar la red neuronal.

In [ ]:
mlp_sin_escalar = MLPClassifier(
    hidden_layer_sizes=(100, 50), activation='relu', max_iter=300, random_state=42
)
mlp_con_escalar = MLPClassifier(
    hidden_layer_sizes=(100, 50), activation='relu', max_iter=300, random_state=42
)

mlp_sin_escalar.fit(X_train, y_train)  # Sin normalizar
mlp_con_escalar.fit(X_train_sc, y_train)  # Con normalización

acc_sin = accuracy_score(y_test, mlp_sin_escalar.predict(X_test))
acc_con = accuracy_score(y_test, mlp_con_escalar.predict(X_test_sc))

print(f'MLP sin normalización: {acc_sin:.2%}')
print(f'MLP con normalización: {acc_con:.2%}')
print(f'Diferencia: {acc_con - acc_sin:+.2%}')
print()
print('¿Por qué importa normalizar?')
print('La variable "edad" va de 18 a 35 (rango ~17)')
print('La variable "asistencia" va de 0.55 a 0.99 (rango ~0.44)')
print('Sin normalizar, la edad domina los cálculos solo por su escala mayor.')

## 8. Vista previa de Keras — el siguiente nivel

Scikit-learn es perfecto para empezar. Cuando los proyectos se vuelven más complejos, se usa **Keras/TensorFlow** o **PyTorch**. Aquí vemos cómo se vería la misma red en Keras:

In [ ]:
# Este código muestra la sintaxis de Keras — no es necesario ejecutarlo
# Si tienes tensorflow instalado, puedes probar: pip install tensorflow

keras_codigo = '''
from tensorflow import keras

# Definir la arquitectura (igual que hidden_layer_sizes=(100, 50))
modelo_keras = keras.Sequential([
    keras.layers.Dense(100, activation='relu', input_shape=(4,)),
    keras.layers.Dense(50, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

# Configurar el entrenamiento
modelo_keras.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Entrenar
history = modelo_keras.fit(
    X_train_sc, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)
'''

print('Código equivalente en Keras:')
print(keras_codigo)
print()
print('Diferencias principales entre sklearn.MLPClassifier y Keras:')
print('- Keras es más flexible: puedes personalizar cada capa')
print('- Keras permite usar GPU para entrenar más rápido')
print('- Keras muestra la precisión en validación durante el entrenamiento')
print('- Keras es necesario para redes convolucionales (imágenes), recurrentes (texto), etc.')

## Resumen

### Lo que aprendimos hoy:

1. **Una neurona** calcula `z = Σ(xi × wi) + b` y aplica una función de activación
2. **ReLU** es la función de activación más común en capas ocultas: `max(0, z)`
3. **Las redes aprenden** ajustando los pesos para minimizar el error (backpropagation)
4. **MLPClassifier** de scikit-learn es la forma más sencilla de usar redes neuronales
5. **Normalizar siempre** antes de entrenar redes neuronales
6. **En datos tabulares pequeños**, Random Forest suele ser igual o mejor que MLP
7. **Keras/TensorFlow** es el siguiente paso para redes más complejas

### ¿Cuándo usar MLP?
- Cuando tienes muchos datos y los patrones son complejos
- Cuando otros modelos han alcanzado su límite
- Cuando trabajas con imágenes, texto o audio (con arquitecturas más avanzadas)